# Chest X-Ray Pneumonia Detection - Phase 2
## Segmentation, Feature Extraction, Classification & Clustering
### Digital Image Processing Project - Spring 2026
---

This notebook implements Phase 2 of the chest X-ray pneumonia detection project:

1. **Segmentation** - Otsu's thresholding (from scratch) for lung ROI extraction
2. **Feature Extraction** - LBP & GLCM (from scratch) + HOG + CNN features (VGG16, VGG19, ResNet50, EfficientNet-B0)
3. **Feature Fusion** - Combining CNN and handcrafted features
4. **Dimensionality Reduction** - PCA (from scratch)
5. **Classification** - KNN (from scratch) + SVM, Random Forest, XGBoost comparison
6. **Clustering** - K-Means (from scratch)
7. **Comprehensive Evaluation** - Accuracy, Sensitivity, Specificity, Precision, Recall, F1, AUC-ROC, Kappa

## 0. Setup & Data Loading

In [ ]:
import os
import time
import gc
import warnings
import pickle

import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

import torch
import torch.nn as nn
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, cohen_kappa_score
from sklearn.manifold import TSNE
from skimage.feature import hog

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 100

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print('All libraries imported successfully!')

In [ ]:
# === PATH CONFIGURATION ===
PREPROCESSED_DIR = r'D:\CSIT\final year\Image\project\archive\chest_xray_preprocessed'
PROJECT_DIR = r'D:\CSIT\final year\Image\project'
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
FIGURES_DIR = os.path.join(PROJECT_DIR, 'figures')

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

TRAIN_DIR = os.path.join(PREPROCESSED_DIR, 'train')
TEST_DIR = os.path.join(PREPROCESSED_DIR, 'test')
VAL_DIR = os.path.join(PREPROCESSED_DIR, 'val')

IMG_SIZE = 256

print(f'Preprocessed data: {PREPROCESSED_DIR}')
print(f'Path exists: {os.path.exists(PREPROCESSED_DIR)}')

In [ ]:
def load_file_list(directory):
    """Load file paths and labels without loading images into memory."""
    files, labels = [], []
    for label_idx, class_name in enumerate(['NORMAL', 'PNEUMONIA']):
        class_dir = os.path.join(directory, class_name)
        if not os.path.exists(class_dir):
            continue
        for fname in sorted(os.listdir(class_dir)):
            if fname.lower().endswith(('.jpeg', '.jpg', '.png')):
                files.append(os.path.join(class_dir, fname))
                labels.append(label_idx)
    return files, np.array(labels)

def load_image(filepath, target_size=256):
    """Load a single grayscale image and resize."""
    img = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f'Could not load: {filepath}')
    if img.shape[0] != target_size or img.shape[1] != target_size:
        img = cv2.resize(img, (target_size, target_size))
    return img

# Load train + val (merged) and test
train_files, train_labels = load_file_list(TRAIN_DIR)
val_files, val_labels = load_file_list(VAL_DIR)
test_files, test_labels = load_file_list(TEST_DIR)

# Merge validation into training (only 16 images - too few to be useful)
train_files = train_files + val_files
train_labels = np.concatenate([train_labels, val_labels])

print(f'Training: {len(train_files)} images (Normal: {np.sum(train_labels==0)}, Pneumonia: {np.sum(train_labels==1)})')
print(f'Testing:  {len(test_files)} images (Normal: {np.sum(test_labels==0)}, Pneumonia: {np.sum(test_labels==1)})')
print(f'Class ratio (Normal:Pneumonia) = 1:{np.sum(train_labels==1)/np.sum(train_labels==0):.2f}')

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2ecc71', '#e74c3c']

for ax, (title, labels) in zip(axes, [('Training Set', train_labels), ('Test Set', test_labels)]):
    counts = [np.sum(labels == 0), np.sum(labels == 1)]
    bars = ax.bar(['Normal', 'Pneumonia'], counts, color=colors, edgecolor='black')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_ylabel('Number of Images')
    for bar, val in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 10,
                str(val), ha='center', va='bottom', fontweight='bold')

plt.suptitle('Dataset Distribution (Phase 2)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 1. Lung ROI Segmentation Using Otsu's Thresholding (From Scratch)

Otsu's method finds the optimal threshold that minimizes intra-class variance by maximizing inter-class variance:

$\sigma_B^2(t) = w_0(t) \cdot w_1(t) \cdot [\mu_0(t) - \mu_1(t)]^2$

where $w_0, w_1$ are class probabilities and $\mu_0, \mu_1$ are class means.

In [ ]:
def otsu_threshold(image):
    """
    Compute Otsu's optimal threshold from scratch using numpy only.
    Maximizes inter-class variance across all possible thresholds.
    """
    # Compute normalized histogram
    hist = np.zeros(256, dtype=np.float64)
    for pixel in image.ravel():
        hist[pixel] += 1
    hist = hist / image.size
    
    best_threshold = 0
    best_variance = 0.0
    
    for t in range(1, 256):
        # Class probabilities
        w0 = np.sum(hist[:t])
        w1 = np.sum(hist[t:])
        
        if w0 == 0 or w1 == 0:
            continue
        
        # Class means
        mu0 = np.sum(np.arange(t) * hist[:t]) / w0
        mu1 = np.sum(np.arange(t, 256) * hist[t:]) / w1
        
        # Inter-class variance
        variance = w0 * w1 * (mu0 - mu1) ** 2
        
        if variance > best_variance:
            best_variance = variance
            best_threshold = t
    
    return best_threshold

# Test on a sample image
sample_img = load_image(train_files[0])
threshold = otsu_threshold(sample_img)
print(f'Otsu threshold (from scratch): {threshold}')

# Validate against OpenCV
cv2_thresh, _ = cv2.threshold(sample_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print(f'OpenCV Otsu threshold: {int(cv2_thresh)}')
print(f'Difference: {abs(threshold - int(cv2_thresh))} (<=1 is expected due to boundary handling)')
print(f'Match: {abs(threshold - int(cv2_thresh)) <= 1}')

In [ ]:
def segment_lung_roi(image):
    """Apply Otsu segmentation and morphological cleanup to extract lung ROI."""
    threshold = otsu_threshold(image)
    mask = (image > threshold).astype(np.uint8)
    
    # Morphological cleanup
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    
    roi = image * mask
    return roi, mask

In [ ]:
# Visualize segmentation on sample images
sample_indices = [0, len(train_files)//4, len(train_files)//2, len(train_files)*3//4]
fig, axes = plt.subplots(4, 4, figsize=(20, 20))

for row, idx in enumerate(sample_indices):
    img = load_image(train_files[idx])
    label_name = 'Normal' if train_labels[idx] == 0 else 'Pneumonia'
    threshold = otsu_threshold(img)
    roi, mask = segment_lung_roi(img)
    
    axes[row, 0].imshow(img, cmap='gray')
    axes[row, 0].set_title(f'Original ({label_name})')
    axes[row, 0].axis('off')
    
    axes[row, 1].hist(img.ravel(), bins=256, range=(0, 256), color='steelblue', alpha=0.7)
    axes[row, 1].axvline(x=threshold, color='red', linewidth=2, label=f'Otsu T={threshold}')
    axes[row, 1].legend()
    axes[row, 1].set_title('Histogram + Otsu Threshold')
    
    axes[row, 2].imshow(mask * 255, cmap='gray')
    axes[row, 2].set_title('Binary Mask')
    axes[row, 2].axis('off')
    
    axes[row, 3].imshow(roi, cmap='gray')
    axes[row, 3].set_title('Segmented ROI')
    axes[row, 3].axis('off')

plt.suptitle('Otsu Thresholding Segmentation (From Scratch)', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'segmentation_otsu.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 2. Handcrafted Feature Extraction
### 2.1 Local Binary Pattern (LBP) - From Scratch

LBP encodes local texture by comparing each pixel with its neighbors and forming a binary number.

In [ ]:
def compute_lbp(image, radius=1, n_points=8):
    """
    Compute Local Binary Pattern from scratch using numpy.
    Vectorized implementation for efficiency.
    """
    rows, cols = image.shape
    lbp_image = np.zeros((rows - 2 * radius, cols - 2 * radius), dtype=np.uint8)
    center = image[radius:rows - radius, radius:cols - radius].astype(np.int16)
    
    for k in range(n_points):
        angle = 2 * np.pi * k / n_points
        di = int(round(-radius * np.sin(angle)))
        dj = int(round(radius * np.cos(angle)))
        neighbor = image[radius + di:rows - radius + di,
                         radius + dj:cols - radius + dj].astype(np.int16)
        lbp_image += ((neighbor >= center).astype(np.uint8)) << k
    
    # Compute normalized histogram as feature vector
    hist = np.bincount(lbp_image.ravel(), minlength=256).astype(np.float64)
    hist /= (hist.sum() + 1e-7)
    return lbp_image, hist

# Test on sample
sample_img = load_image(train_files[0])
lbp_img, lbp_hist = compute_lbp(sample_img)
print(f'LBP image shape: {lbp_img.shape}')
print(f'LBP histogram length: {len(lbp_hist)}')

In [ ]:
# Visualize LBP for Normal vs Pneumonia
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

normal_idx = np.where(train_labels == 0)[0][0]
pneumonia_idx = np.where(train_labels == 1)[0][0]

for row, idx, label in [(0, normal_idx, 'Normal'), (1, pneumonia_idx, 'Pneumonia')]:
    img = load_image(train_files[idx])
    lbp_img, lbp_hist = compute_lbp(img)
    
    axes[row, 0].imshow(img, cmap='gray')
    axes[row, 0].set_title(f'Original ({label})')
    axes[row, 0].axis('off')
    
    axes[row, 1].imshow(lbp_img, cmap='gray')
    axes[row, 1].set_title(f'LBP Image ({label})')
    axes[row, 1].axis('off')
    
    axes[row, 2].bar(range(256), lbp_hist, color='steelblue', alpha=0.7)
    axes[row, 2].set_title(f'LBP Histogram ({label})')
    axes[row, 2].set_xlabel('LBP Value')
    axes[row, 2].set_ylabel('Normalized Frequency')

plt.suptitle('Local Binary Pattern (LBP) - From Scratch', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'lbp_visualization.png'), dpi=150, bbox_inches='tight')
plt.show()

### 2.2 Gray-Level Co-occurrence Matrix (GLCM) - From Scratch

GLCM captures spatial relationships between pixel intensities, providing texture descriptors:
Contrast, Dissimilarity, Homogeneity, Energy, Correlation, Entropy.

In [ ]:
def compute_glcm(image, d=1, angle=0, levels=32):
    """
    Compute Gray-Level Co-occurrence Matrix from scratch.
    Vectorized implementation using numpy.
    """
    # Quantize to fewer levels
    quantized = np.clip((image // (256 // levels)).astype(np.int32), 0, levels - 1)
    
    dx = int(round(d * np.cos(angle)))
    dy = int(round(-d * np.sin(angle)))
    
    rows, cols = quantized.shape
    r1 = quantized[max(0, -dy):min(rows, rows - dy), max(0, -dx):min(cols, cols - dx)]
    r2 = quantized[max(0, dy):min(rows, rows + dy), max(0, dx):min(cols, cols + dx)]
    
    # Ensure same shape
    min_rows = min(r1.shape[0], r2.shape[0])
    min_cols = min(r1.shape[1], r2.shape[1])
    r1 = r1[:min_rows, :min_cols]
    r2 = r2[:min_rows, :min_cols]
    
    glcm = np.zeros((levels, levels), dtype=np.float64)
    np.add.at(glcm, (r1.ravel(), r2.ravel()), 1)
    
    # Make symmetric
    glcm = (glcm + glcm.T) / 2.0
    total = glcm.sum()
    if total > 0:
        glcm /= total
    return glcm


def glcm_features(glcm):
    """Extract 6 texture features from a GLCM matrix."""
    levels = glcm.shape[0]
    i_idx, j_idx = np.meshgrid(np.arange(levels, dtype=np.float64),
                                np.arange(levels, dtype=np.float64), indexing='ij')
    
    contrast = np.sum(glcm * (i_idx - j_idx) ** 2)
    dissimilarity = np.sum(glcm * np.abs(i_idx - j_idx))
    homogeneity = np.sum(glcm / (1.0 + (i_idx - j_idx) ** 2))
    energy = np.sum(glcm ** 2)
    
    mu_i = np.sum(i_idx * glcm)
    mu_j = np.sum(j_idx * glcm)
    sigma_i = np.sqrt(np.sum(glcm * (i_idx - mu_i) ** 2))
    sigma_j = np.sqrt(np.sum(glcm * (j_idx - mu_j) ** 2))
    if sigma_i > 1e-10 and sigma_j > 1e-10:
        correlation = np.sum(glcm * (i_idx - mu_i) * (j_idx - mu_j)) / (sigma_i * sigma_j)
    else:
        correlation = 0.0
    
    nonzero = glcm[glcm > 0]
    entropy = -np.sum(nonzero * np.log2(nonzero))
    
    return np.array([contrast, dissimilarity, homogeneity, energy, correlation, entropy])


# Test
sample_glcm = compute_glcm(sample_img, d=1, angle=0, levels=32)
feats = glcm_features(sample_glcm)
print('GLCM features: Contrast={:.2f}, Dissimilarity={:.2f}, Homogeneity={:.4f}, Energy={:.4f}, Correlation={:.4f}, Entropy={:.2f}'.format(*feats))

In [ ]:
# Visualize GLCM matrices for Normal vs Pneumonia
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
angle_names = ['0°', '45°', '90°', '135°']
angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]

for row, idx, label in [(0, normal_idx, 'Normal'), (1, pneumonia_idx, 'Pneumonia')]:
    img = load_image(train_files[idx])
    for col, (angle, aname) in enumerate(zip(angles, angle_names)):
        glcm = compute_glcm(img, d=1, angle=angle, levels=32)
        axes[row, col].imshow(np.log1p(glcm), cmap='hot')
        axes[row, col].set_title(f'{label} - {aname}')
        axes[row, col].axis('off')

plt.suptitle('GLCM Matrices at Different Angles (From Scratch)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'glcm_visualization.png'), dpi=150, bbox_inches='tight')
plt.show()

### 2.3 Histogram of Oriented Gradients (HOG)

In [ ]:
def compute_hog_features(image):
    """Compute HOG features using skimage."""
    features, hog_image = hog(image, orientations=9, pixels_per_cell=(32, 32),
                              cells_per_block=(2, 2), visualize=True, feature_vector=True)
    return features, hog_image

# Visualize HOG for Normal vs Pneumonia
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for row, idx, label in [(0, normal_idx, 'Normal'), (1, pneumonia_idx, 'Pneumonia')]:
    img = load_image(train_files[idx])
    hog_feats, hog_img = compute_hog_features(img)
    
    axes[row, 0].imshow(img, cmap='gray')
    axes[row, 0].set_title(f'Original ({label})')
    axes[row, 0].axis('off')
    
    axes[row, 1].imshow(hog_img, cmap='gray')
    axes[row, 1].set_title(f'HOG Image ({label}) - {len(hog_feats)} features')
    axes[row, 1].axis('off')

plt.suptitle('Histogram of Oriented Gradients (HOG)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'hog_visualization.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'HOG feature vector length: {len(hog_feats)}')

### 2.4 Batch Handcrafted Feature Extraction

In [ ]:
def extract_handcrafted_features(image):
    """Extract all handcrafted features from a single image."""
    # Segment ROI
    roi, _ = segment_lung_roi(image)
    
    # LBP: 256-dim histogram
    _, lbp_hist = compute_lbp(roi)
    
    # GLCM: 6 features x 4 angles = 24 features
    glcm_feats = []
    for angle in [0, np.pi/4, np.pi/2, 3*np.pi/4]:
        glcm = compute_glcm(roi, d=1, angle=angle, levels=32)
        glcm_feats.append(glcm_features(glcm))
    glcm_feats = np.concatenate(glcm_feats)
    
    # HOG features
    hog_feats, _ = compute_hog_features(roi)
    
    return np.concatenate([lbp_hist, glcm_feats, hog_feats]).astype(np.float32)

# Test on one image to get feature dimensions
test_feat = extract_handcrafted_features(load_image(train_files[0]))
print(f'Total handcrafted features per image: {len(test_feat)}')
print(f'  LBP histogram: 256')
print(f'  GLCM features: 24 (6 x 4 angles)')
print(f'  HOG features: {len(test_feat) - 256 - 24}')

In [ ]:
# Extract handcrafted features for all images
hc_train_path = os.path.join(RESULTS_DIR, 'handcrafted_train.npy')
hc_test_path = os.path.join(RESULTS_DIR, 'handcrafted_test.npy')

if os.path.exists(hc_train_path) and os.path.exists(hc_test_path):
    print('Loading cached handcrafted features...')
    X_train_hc = np.load(hc_train_path)
    X_test_hc = np.load(hc_test_path)
else:
    print('Extracting handcrafted features (training set)...')
    start = time.time()
    X_train_hc = []
    for i, fpath in enumerate(train_files):
        img = load_image(fpath)
        feat = extract_handcrafted_features(img)
        X_train_hc.append(feat)
        if (i + 1) % 500 == 0:
            elapsed = time.time() - start
            print(f'  {i+1}/{len(train_files)} images ({elapsed:.1f}s)')
    X_train_hc = np.array(X_train_hc)
    print(f'  Training done in {time.time()-start:.1f}s')
    
    print('Extracting handcrafted features (test set)...')
    start = time.time()
    X_test_hc = []
    for i, fpath in enumerate(test_files):
        img = load_image(fpath)
        feat = extract_handcrafted_features(img)
        X_test_hc.append(feat)
    X_test_hc = np.array(X_test_hc)
    print(f'  Test done in {time.time()-start:.1f}s')
    
    np.save(hc_train_path, X_train_hc)
    np.save(hc_test_path, X_test_hc)
    print('Features saved to disk.')

print(f'Handcrafted features - Train: {X_train_hc.shape}, Test: {X_test_hc.shape}')

---
## 3. CNN Feature Extraction (Transfer Learning)

Extract deep features from pretrained CNN architectures: VGG16, VGG19, ResNet50, EfficientNet-B0.
Using Global Average Pooling to produce compact feature vectors.

In [ ]:
class CNNFeatureExtractor:
    """Extract features from pretrained CNN models using PyTorch."""
    
    MODEL_CONFIGS = {
        'vgg16': {'constructor': models.vgg16, 'weights': models.VGG16_Weights.IMAGENET1K_V1, 'feat_attr': 'features', 'dim': 512},
        'vgg19': {'constructor': models.vgg19, 'weights': models.VGG19_Weights.IMAGENET1K_V1, 'feat_attr': 'features', 'dim': 512},
        'resnet50': {'constructor': models.resnet50, 'weights': models.ResNet50_Weights.IMAGENET1K_V1, 'feat_attr': None, 'dim': 2048},
        'efficientnet_b0': {'constructor': models.efficientnet_b0, 'weights': models.EfficientNet_B0_Weights.IMAGENET1K_V1, 'feat_attr': 'features', 'dim': 1280},
    }
    
    def __init__(self, model_name):
        config = self.MODEL_CONFIGS[model_name]
        self.model_name = model_name
        self.feature_dim = config['dim']
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        base_model = config['constructor'](weights=config['weights'])
        
        if model_name == 'resnet50':
            layers = list(base_model.children())[:-1]  # Remove final FC
            self.model = nn.Sequential(*layers, nn.Flatten())
        else:
            feat_layers = getattr(base_model, config['feat_attr'])
            self.model = nn.Sequential(feat_layers, nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten())
        
        self.model.eval()
        self.model.to(self.device)
        
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Lambda(lambda x: x.repeat(3, 1, 1) if x.shape[0] == 1 else x),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    
    def extract_batch(self, file_list, batch_size=16):
        all_features = []
        for i in range(0, len(file_list), batch_size):
            batch_files = file_list[i:i + batch_size]
            batch_tensors = []
            for fpath in batch_files:
                img = load_image(fpath)
                tensor = self.transform(img)
                batch_tensors.append(tensor)
            
            batch = torch.stack(batch_tensors).to(self.device)
            with torch.no_grad():
                features = self.model(batch)
            all_features.append(features.cpu().numpy().astype(np.float32))
            
            if (i + batch_size) % 500 < batch_size:
                print(f'    {self.model_name}: {min(i + batch_size, len(file_list))}/{len(file_list)}')
        
        return np.vstack(all_features)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'CNNFeatureExtractor defined. Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB)')
    print(f'Estimated CNN extraction time: ~8-10 minutes total')
else:
    print(f'Running on CPU. Estimated CNN extraction time: ~60-70 minutes total')
print('Models available:', list(CNNFeatureExtractor.MODEL_CONFIGS.keys()))

In [ ]:
# Extract features from all CNN architectures
cnn_models = ['vgg16', 'vgg19', 'resnet50', 'efficientnet_b0']
cnn_features_train = {}
cnn_features_test = {}
cnn_times = {}

for model_name in cnn_models:
    train_path = os.path.join(RESULTS_DIR, f'cnn_{model_name}_train.npy')
    test_path = os.path.join(RESULTS_DIR, f'cnn_{model_name}_test.npy')
    
    if os.path.exists(train_path) and os.path.exists(test_path):
        print(f'{model_name}: Loading cached features...')
        cnn_features_train[model_name] = np.load(train_path)
        cnn_features_test[model_name] = np.load(test_path)
        cnn_times[model_name] = 0
    else:
        print(f'\n{model_name}: Extracting features...')
        start = time.time()
        extractor = CNNFeatureExtractor(model_name)
        
        print(f'  Training set ({len(train_files)} images)...')
        cnn_features_train[model_name] = extractor.extract_batch(train_files)
        
        print(f'  Test set ({len(test_files)} images)...')
        cnn_features_test[model_name] = extractor.extract_batch(test_files)
        
        elapsed = time.time() - start
        cnn_times[model_name] = elapsed
        print(f'  Done in {elapsed:.1f}s')
        
        np.save(train_path, cnn_features_train[model_name])
        np.save(test_path, cnn_features_test[model_name])
        
        del extractor
        gc.collect()

print('\n=== CNN Feature Extraction Summary ===')
for name in cnn_models:
    print(f'  {name:20s}: {cnn_features_train[name].shape[1]:5d} features, {cnn_times[name]:.1f}s')

In [ ]:
# Quick comparison: Which CNN architecture produces the best features?
print('=== CNN Architecture Comparison (SVM classifier) ===')
cnn_accuracies = {}

for model_name in cnn_models:
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(cnn_features_train[model_name])
    X_te = scaler.transform(cnn_features_test[model_name])
    
    svm = SVC(kernel='rbf', class_weight='balanced', random_state=42)
    svm.fit(X_tr, train_labels)
    acc = svm.score(X_te, test_labels)
    cnn_accuracies[model_name] = acc
    print(f'  {model_name:20s}: Accuracy = {acc:.4f}')

best_cnn = max(cnn_accuracies, key=cnn_accuracies.get)
print(f'\nBest CNN architecture: {best_cnn} ({cnn_accuracies[best_cnn]:.4f})')

# Bar chart
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(cnn_accuracies.keys(), cnn_accuracies.values(), color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12'], edgecolor='black')
ax.set_ylabel('Accuracy')
ax.set_title('CNN Architecture Comparison (SVM Classifier)', fontsize=14, fontweight='bold')
ax.set_ylim([0.5, 1.0])
for bar, val in zip(bars, cnn_accuracies.values()):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'cnn_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Feature Fusion & PCA (From Scratch)
### 4.1 Feature Fusion

In [ ]:
# Standardize and fuse best CNN features with handcrafted features
scaler_cnn = StandardScaler()
cnn_train_scaled = scaler_cnn.fit_transform(cnn_features_train[best_cnn])
cnn_test_scaled = scaler_cnn.transform(cnn_features_test[best_cnn])

scaler_hc = StandardScaler()
hc_train_scaled = scaler_hc.fit_transform(X_train_hc)
hc_test_scaled = scaler_hc.transform(X_test_hc)

# Early fusion: concatenate
X_train_fused = np.hstack([cnn_train_scaled, hc_train_scaled]).astype(np.float32)
X_test_fused = np.hstack([cnn_test_scaled, hc_test_scaled]).astype(np.float32)

print(f'CNN features ({best_cnn}): {cnn_train_scaled.shape[1]} dims')
print(f'Handcrafted features: {hc_train_scaled.shape[1]} dims')
print(f'Fused features: {X_train_fused.shape[1]} dims')
print(f'Train shape: {X_train_fused.shape}, Test shape: {X_test_fused.shape}')

In [ ]:
# t-SNE visualization of features
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

n_vis = 1000  # subsample for t-SNE speed
vis_idx = np.random.choice(len(train_labels), min(n_vis, len(train_labels)), replace=False)
vis_labels = train_labels[vis_idx]
colors_map = {0: '#2ecc71', 1: '#e74c3c'}
vis_colors = [colors_map[l] for l in vis_labels]

feature_sets = [
    ('Handcrafted Only', hc_train_scaled[vis_idx]),
    (f'CNN Only ({best_cnn})', cnn_train_scaled[vis_idx]),
    ('Fused (CNN + Handcrafted)', X_train_fused[vis_idx]),
]

for ax, (title, feats) in zip(axes, feature_sets):
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    embedded = tsne.fit_transform(feats)
    ax.scatter(embedded[vis_labels == 0, 0], embedded[vis_labels == 0, 1],
               c='#2ecc71', label='Normal', alpha=0.5, s=10)
    ax.scatter(embedded[vis_labels == 1, 0], embedded[vis_labels == 1, 1],
               c='#e74c3c', label='Pneumonia', alpha=0.5, s=10)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend()

plt.suptitle('t-SNE Feature Visualization', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'feature_tsne.png'), dpi=150, bbox_inches='tight')
plt.show()

### 4.2 PCA - Principal Component Analysis (From Scratch)

In [ ]:
class PCA_Scratch:
    """PCA implementation from scratch using numpy eigendecomposition."""
    
    def __init__(self, n_components):
        self.n_components = n_components
        self.components = None
        self.mean = None
        self.explained_variance = None
        self.explained_variance_ratio = None
    
    def fit(self, X):
        X = X.astype(np.float64)
        self.mean = np.mean(X, axis=0)
        X_centered = X - self.mean
        
        n_samples = X_centered.shape[0]
        cov_matrix = np.dot(X_centered.T, X_centered) / (n_samples - 1)
        
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
        
        idx = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]
        
        # Ensure non-negative eigenvalues
        eigenvalues = np.maximum(eigenvalues, 0)
        
        self.components = eigenvectors[:, :self.n_components].T
        self.explained_variance = eigenvalues[:self.n_components]
        total_var = eigenvalues.sum()
        self.explained_variance_ratio = eigenvalues[:self.n_components] / total_var if total_var > 0 else np.zeros(self.n_components)
        self.cumulative_variance_ratio = np.cumsum(eigenvalues / total_var) if total_var > 0 else np.zeros(len(eigenvalues))
        
        return self
    
    def transform(self, X):
        X_centered = X.astype(np.float64) - self.mean
        return np.dot(X_centered, self.components.T).astype(np.float32)
    
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

print('PCA_Scratch class defined.')

In [ ]:
# Find optimal number of components using explained variance
max_components = min(X_train_fused.shape[0], X_train_fused.shape[1])
pca_full = PCA_Scratch(n_components=max_components)
pca_full.fit(X_train_fused)

cumvar = pca_full.cumulative_variance_ratio

# Find components for 95% and 99% variance
n_95 = np.searchsorted(cumvar, 0.95) + 1
n_99 = np.searchsorted(cumvar, 0.99) + 1

print(f'Components for 95% variance: {n_95}')
print(f'Components for 99% variance: {n_99}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(range(1, min(51, len(pca_full.explained_variance_ratio)+1)),
             pca_full.explained_variance_ratio[:50], 'bo-', markersize=4)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Individual Explained Variance')

axes[1].plot(range(1, len(cumvar)+1), cumvar, 'r-', linewidth=2)
axes[1].axhline(y=0.95, color='green', linestyle='--', label=f'95% ({n_95} components)')
axes[1].axhline(y=0.99, color='blue', linestyle='--', label=f'99% ({n_99} components)')
axes[1].axvline(x=n_95, color='green', linestyle=':', alpha=0.5)
axes[1].axvline(x=n_99, color='blue', linestyle=':', alpha=0.5)
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()
axes[1].set_xlim([0, min(500, len(cumvar))])

plt.suptitle('PCA Analysis (From Scratch)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'pca_variance.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Apply PCA with 95% variance threshold
n_components = n_95
pca = PCA_Scratch(n_components=n_components)
X_train_pca = pca.fit_transform(X_train_fused)
X_test_pca = pca.transform(X_test_fused)

print(f'PCA reduction: {X_train_fused.shape[1]} -> {n_components} features')
print(f'Variance retained: {sum(pca.explained_variance_ratio)*100:.2f}%')
print(f'Train shape: {X_train_pca.shape}, Test shape: {X_test_pca.shape}')

# Validate against sklearn PCA
from sklearn.decomposition import PCA as SklearnPCA
sklearn_pca = SklearnPCA(n_components=n_components)
X_train_sklearn = sklearn_pca.fit_transform(X_train_fused)

print(f'\n=== Validation: Scratch vs sklearn PCA ===')
print(f'Explained variance ratio (first 5):')
print(f'  Scratch: {pca.explained_variance_ratio[:5]}')
print(f'  sklearn: {sklearn_pca.explained_variance_ratio_[:5]}')
print(f'  Match: {np.allclose(pca.explained_variance_ratio[:5], sklearn_pca.explained_variance_ratio_[:5], atol=1e-4)}')

---
## 5. KNN Classification (From Scratch)

In [ ]:
class KNN_Scratch:
    """K-Nearest Neighbors classifier from scratch using numpy."""
    
    def __init__(self, k=5):
        self.k = k
        self.X_train = None
        self.y_train = None
    
    def fit(self, X, y):
        self.X_train = X.astype(np.float64)
        self.y_train = y.copy()
        return self
    
    def _compute_distances(self, X_query):
        """Compute Euclidean distances between query points and training points."""
        # Vectorized: ||a - b||^2 = ||a||^2 + ||b||^2 - 2*a.b
        X_q = X_query.astype(np.float64)
        sq_train = np.sum(self.X_train ** 2, axis=1)
        sq_query = np.sum(X_q ** 2, axis=1)
        cross = np.dot(X_q, self.X_train.T)
        distances = np.sqrt(np.maximum(sq_query[:, np.newaxis] + sq_train[np.newaxis, :] - 2 * cross, 0))
        return distances
    
    def predict(self, X):
        distances = self._compute_distances(X)
        k_indices = np.argsort(distances, axis=1)[:, :self.k]
        k_labels = self.y_train[k_indices]
        predictions = np.array([np.argmax(np.bincount(row.astype(int), minlength=2)) for row in k_labels])
        return predictions
    
    def predict_proba(self, X):
        """Return probability estimates."""
        distances = self._compute_distances(X)
        k_indices = np.argsort(distances, axis=1)[:, :self.k]
        k_labels = self.y_train[k_indices]
        probas = np.array([np.bincount(row.astype(int), minlength=2) / self.k for row in k_labels])
        return probas
    
    def score(self, X, y):
        return np.mean(self.predict(X) == y)

print('KNN_Scratch class defined.')

In [ ]:
# Find optimal K
k_values = [1, 3, 5, 7, 9, 11, 15, 21]
k_accuracies = []

print('=== Finding Optimal K ===')
for k in k_values:
    knn = KNN_Scratch(k=k)
    knn.fit(X_train_pca, train_labels)
    acc = knn.score(X_test_pca, test_labels)
    k_accuracies.append(acc)
    print(f'  K={k:2d}: Accuracy = {acc:.4f}')

best_k = k_values[np.argmax(k_accuracies)]
print(f'\nBest K = {best_k} (Accuracy = {max(k_accuracies):.4f})')

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(k_values, k_accuracies, 'bo-', linewidth=2, markersize=8)
ax.axvline(x=best_k, color='red', linestyle='--', label=f'Best K={best_k}')
ax.set_xlabel('K (Number of Neighbors)', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('KNN: K Selection (From Scratch)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'knn_k_selection.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Train final KNN with best K
knn_best = KNN_Scratch(k=best_k)
knn_best.fit(X_train_pca, train_labels)
knn_pred = knn_best.predict(X_test_pca)
knn_proba = knn_best.predict_proba(X_test_pca)

print(f'KNN (K={best_k}) Test Accuracy: {np.mean(knn_pred == test_labels):.4f}')
print(f'\nClassification Report:')
print(classification_report(test_labels, knn_pred, target_names=['Normal', 'Pneumonia']))

---
## 6. K-Means Clustering (From Scratch)

In [ ]:
class KMeans_Scratch:
    """K-Means clustering from scratch with K-Means++ initialization."""
    
    def __init__(self, n_clusters=2, max_iter=300, tol=1e-4, random_state=42):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state
        self.centroids = None
        self.labels = None
        self.inertia = None
        self.inertia_history = []
    
    def _init_centroids(self, X):
        """K-Means++ initialization."""
        rng = np.random.RandomState(self.random_state)
        n_samples = X.shape[0]
        centroids = [X[rng.randint(n_samples)]]
        
        for _ in range(1, self.n_clusters):
            distances = np.min([np.sum((X - c) ** 2, axis=1) for c in centroids], axis=0)
            probs = distances / distances.sum()
            centroids.append(X[rng.choice(n_samples, p=probs)])
        
        return np.array(centroids)
    
    def fit(self, X):
        X = X.astype(np.float64)
        self.centroids = self._init_centroids(X)
        self.inertia_history = []
        
        for iteration in range(self.max_iter):
            # Assign clusters
            distances = np.array([np.sum((X - c) ** 2, axis=1) for c in self.centroids])
            self.labels = np.argmin(distances, axis=0)
            
            # Update centroids
            new_centroids = np.array([
                X[self.labels == k].mean(axis=0) if np.sum(self.labels == k) > 0
                else self.centroids[k]
                for k in range(self.n_clusters)
            ])
            
            # Compute inertia
            self.inertia = sum(np.sum((X[self.labels == k] - new_centroids[k]) ** 2)
                               for k in range(self.n_clusters))
            self.inertia_history.append(self.inertia)
            
            # Check convergence
            if np.sum((new_centroids - self.centroids) ** 2) < self.tol:
                print(f'  Converged at iteration {iteration + 1}')
                break
            self.centroids = new_centroids
        
        return self
    
    def predict(self, X):
        X = X.astype(np.float64)
        distances = np.array([np.sum((X - c) ** 2, axis=1) for c in self.centroids])
        return np.argmin(distances, axis=0)

print('KMeans_Scratch class defined.')

In [ ]:
# Apply K-Means with K=2
kmeans = KMeans_Scratch(n_clusters=2, random_state=42)
kmeans.fit(X_train_pca)

cluster_labels_train = kmeans.labels
cluster_labels_test = kmeans.predict(X_test_pca)

# Map cluster labels to true labels using majority vote
def map_clusters_to_labels(cluster_labels, true_labels, n_clusters):
    """Map cluster assignments to class labels via majority vote."""
    mapping = {}
    for k in range(n_clusters):
        mask = cluster_labels == k
        if mask.sum() > 0:
            counts = np.bincount(true_labels[mask].astype(int), minlength=2)
            mapping[k] = np.argmax(counts)
        else:
            mapping[k] = k
    mapped = np.array([mapping[c] for c in cluster_labels])
    return mapped, mapping

mapped_train, cluster_mapping = map_clusters_to_labels(cluster_labels_train, train_labels, 2)
mapped_test = np.array([cluster_mapping[c] for c in cluster_labels_test])

kmeans_acc_train = np.mean(mapped_train == train_labels)
kmeans_acc_test = np.mean(mapped_test == test_labels)
print(f'K-Means Clustering Accuracy (train): {kmeans_acc_train:.4f}')
print(f'K-Means Clustering Accuracy (test):  {kmeans_acc_test:.4f}')
print(f'Cluster mapping: {cluster_mapping}')

In [ ]:
# Convergence plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(range(1, len(kmeans.inertia_history) + 1), kmeans.inertia_history, 'ro-', linewidth=2)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Inertia')
axes[0].set_title('K-Means Convergence (K=2)', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Elbow method
k_range = range(2, 9)
inertias = []
for k in k_range:
    km = KMeans_Scratch(n_clusters=k, random_state=42)
    km.fit(X_train_pca)
    inertias.append(km.inertia)

axes[1].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Inertia')
axes[1].set_title('Elbow Method', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.suptitle('K-Means Clustering Analysis (From Scratch)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'kmeans_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualize clusters in 2D PCA space
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Plot true labels
axes[0].scatter(X_train_pca[train_labels == 0, 0], X_train_pca[train_labels == 0, 1],
                c='#2ecc71', label='Normal', alpha=0.3, s=10)
axes[0].scatter(X_train_pca[train_labels == 1, 0], X_train_pca[train_labels == 1, 1],
                c='#e74c3c', label='Pneumonia', alpha=0.3, s=10)
axes[0].set_title('True Labels', fontsize=14, fontweight='bold')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].legend()

# Plot cluster labels
axes[1].scatter(X_train_pca[cluster_labels_train == 0, 0], X_train_pca[cluster_labels_train == 0, 1],
                c='#3498db', label='Cluster 0', alpha=0.3, s=10)
axes[1].scatter(X_train_pca[cluster_labels_train == 1, 0], X_train_pca[cluster_labels_train == 1, 1],
                c='#f39c12', label='Cluster 1', alpha=0.3, s=10)
# Plot centroids
axes[1].scatter(kmeans.centroids[:, 0], kmeans.centroids[:, 1],
                c='black', marker='X', s=200, edgecolors='white', linewidths=2, label='Centroids')
axes[1].set_title(f'K-Means Clusters (Acc={kmeans_acc_train:.3f})', fontsize=14, fontweight='bold')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].legend()

plt.suptitle('K-Means Clustering vs True Labels (PCA Space)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'kmeans_clusters.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Comparison with Other ML Classifiers

In [ ]:
def compute_all_metrics(y_true, y_pred, y_prob=None):
    """Compute all required evaluation metrics."""
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = sensitivity
    f_measure = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    kappa = cohen_kappa_score(y_true, y_pred)
    
    auc_roc = None
    if y_prob is not None:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc_roc = auc(fpr, tpr)
    
    return {
        'Accuracy': accuracy,
        'Sensitivity': sensitivity,
        'Specificity': specificity,
        'Precision': precision,
        'Recall': recall,
        'F-measure': f_measure,
        'AUC-ROC': auc_roc,
        'Kappa': kappa,
    }

print('compute_all_metrics function defined.')

In [ ]:
# Define and train all classifiers on fused+PCA features
all_results = {}
all_predictions = {}
all_probabilities = {}

# 1. KNN (from scratch) - already trained
all_predictions['KNN (Scratch)'] = knn_pred
all_probabilities['KNN (Scratch)'] = knn_proba[:, 1]
all_results['KNN (Scratch)'] = compute_all_metrics(test_labels, knn_pred, knn_proba[:, 1])

# 2. K-Means (from scratch)
all_predictions['K-Means (Scratch)'] = mapped_test
all_probabilities['K-Means (Scratch)'] = None
all_results['K-Means (Scratch)'] = compute_all_metrics(test_labels, mapped_test)

# 3. SVM (RBF)
print('Training SVM (RBF)...')
svm_rbf = SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42)
svm_rbf.fit(X_train_pca, train_labels)
svm_rbf_pred = svm_rbf.predict(X_test_pca)
svm_rbf_proba = svm_rbf.predict_proba(X_test_pca)[:, 1]
all_predictions['SVM (RBF)'] = svm_rbf_pred
all_probabilities['SVM (RBF)'] = svm_rbf_proba
all_results['SVM (RBF)'] = compute_all_metrics(test_labels, svm_rbf_pred, svm_rbf_proba)

# 4. SVM (Linear)
print('Training SVM (Linear)...')
svm_lin = SVC(kernel='linear', probability=True, class_weight='balanced', random_state=42)
svm_lin.fit(X_train_pca, train_labels)
svm_lin_pred = svm_lin.predict(X_test_pca)
svm_lin_proba = svm_lin.predict_proba(X_test_pca)[:, 1]
all_predictions['SVM (Linear)'] = svm_lin_pred
all_probabilities['SVM (Linear)'] = svm_lin_proba
all_results['SVM (Linear)'] = compute_all_metrics(test_labels, svm_lin_pred, svm_lin_proba)

# 5. Random Forest
print('Training Random Forest...')
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train_pca, train_labels)
rf_pred = rf.predict(X_test_pca)
rf_proba = rf.predict_proba(X_test_pca)[:, 1]
all_predictions['Random Forest'] = rf_pred
all_probabilities['Random Forest'] = rf_proba
all_results['Random Forest'] = compute_all_metrics(test_labels, rf_pred, rf_proba)

# 6. XGBoost
print('Training XGBoost...')
n_neg = np.sum(train_labels == 0)
n_pos = np.sum(train_labels == 1)
xgb = XGBClassifier(n_estimators=200, scale_pos_weight=n_neg/n_pos,
                     use_label_encoder=False, eval_metric='logloss', random_state=42, verbosity=0)
xgb.fit(X_train_pca, train_labels)
xgb_pred = xgb.predict(X_test_pca)
xgb_proba = xgb.predict_proba(X_test_pca)[:, 1]
all_predictions['XGBoost'] = xgb_pred
all_probabilities['XGBoost'] = xgb_proba
all_results['XGBoost'] = compute_all_metrics(test_labels, xgb_pred, xgb_proba)

print('\nAll classifiers trained successfully!')

---
## 8. Comprehensive Evaluation
### 8.1 Results Table

In [ ]:
# Create comprehensive results table
results_df = pd.DataFrame(all_results).T
results_df = results_df.round(4)

print('=== Classification Results (Fused Features + PCA) ===')
print(results_df.to_string())

# Save as CSV
results_df.to_csv(os.path.join(RESULTS_DIR, 'classification_results.csv'))

# Plot as styled table
fig, ax = plt.subplots(figsize=(16, 4))
ax.axis('off')
table = ax.table(cellText=results_df.values,
                 rowLabels=results_df.index,
                 colLabels=results_df.columns,
                 cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.5)

# Color the header
for j in range(len(results_df.columns)):
    table[0, j].set_facecolor('#3498db')
    table[0, j].set_text_props(color='white', fontweight='bold')

plt.title('Classification Performance Comparison', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'results_table.png'), dpi=150, bbox_inches='tight')
plt.show()

### 8.2 Confusion Matrices

In [ ]:
# Plot confusion matrices for all classifiers
classifiers_with_cm = [name for name in all_predictions.keys()]
n_classifiers = len(classifiers_with_cm)
n_cols = 3
n_rows = (n_classifiers + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
axes = axes.flatten() if n_rows > 1 else [axes] if n_classifiers == 1 else axes.flatten()

for idx, name in enumerate(classifiers_with_cm):
    cm = confusion_matrix(test_labels, all_predictions[name])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Normal', 'Pneumonia'],
                yticklabels=['Normal', 'Pneumonia'])
    acc = all_results[name]['Accuracy']
    axes[idx].set_title(f'{name}\nAcc={acc:.4f}', fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')

# Hide unused axes
for idx in range(n_classifiers, len(axes)):
    axes[idx].axis('off')

plt.suptitle('Confusion Matrices - All Classifiers', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()

### 8.3 ROC Curves

In [ ]:
# Plot ROC curves
fig, ax = plt.subplots(figsize=(10, 8))
colors_roc = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

for idx, (name, proba) in enumerate(all_probabilities.items()):
    if proba is not None:
        fpr, tpr, _ = roc_curve(test_labels, proba)
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=colors_roc[idx % len(colors_roc)], linewidth=2,
                label=f'{name} (AUC = {roc_auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.5)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves - All Classifiers', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'roc_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

### 8.4 CNN Architecture Comparison

In [ ]:
# Compare all CNN architectures using the best classifier
print('=== CNN Architecture Comparison (Full Pipeline) ===')
cnn_comparison = {}

for model_name in cnn_models:
    # Scale CNN features
    scaler_tmp = StandardScaler()
    cnn_tr = scaler_tmp.fit_transform(cnn_features_train[model_name])
    cnn_te = scaler_tmp.transform(cnn_features_test[model_name])
    
    # Fuse with handcrafted
    fused_tr = np.hstack([cnn_tr, hc_train_scaled]).astype(np.float32)
    fused_te = np.hstack([cnn_te, hc_test_scaled]).astype(np.float32)
    
    # PCA
    pca_tmp = PCA_Scratch(n_components=min(n_components, fused_tr.shape[1]))
    pca_tr = pca_tmp.fit_transform(fused_tr)
    pca_te = pca_tmp.transform(fused_te)
    
    # SVM classifier
    svm_tmp = SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42)
    svm_tmp.fit(pca_tr, train_labels)
    pred = svm_tmp.predict(pca_te)
    proba = svm_tmp.predict_proba(pca_te)[:, 1]
    
    cnn_comparison[model_name] = compute_all_metrics(test_labels, pred, proba)

cnn_df = pd.DataFrame(cnn_comparison).T.round(4)
print(cnn_df.to_string())
cnn_df.to_csv(os.path.join(RESULTS_DIR, 'cnn_comparison.csv'))

# Plot
fig, ax = plt.subplots(figsize=(14, 4))
ax.axis('off')
table = ax.table(cellText=cnn_df.values, rowLabels=cnn_df.index,
                 colLabels=cnn_df.columns, cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.5)
for j in range(len(cnn_df.columns)):
    table[0, j].set_facecolor('#e74c3c')
    table[0, j].set_text_props(color='white', fontweight='bold')
plt.title('CNN Architecture Comparison (SVM + Fused Features)', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'cnn_comparison_table.png'), dpi=150, bbox_inches='tight')
plt.show()

### 8.5 Feature Type Comparison

In [ ]:
# Compare: Handcrafted only vs CNN only vs Fused
print('=== Feature Type Comparison (SVM RBF) ===')
feat_comparison = {}

feature_sets_eval = {
    'Handcrafted Only': (hc_train_scaled, hc_test_scaled),
    f'CNN Only ({best_cnn})': (cnn_train_scaled, cnn_test_scaled),
    'Fused (CNN + HC)': (X_train_fused, X_test_fused),
}

for feat_name, (X_tr, X_te) in feature_sets_eval.items():
    # PCA
    n_comp = min(n_components, X_tr.shape[1])
    pca_tmp = PCA_Scratch(n_components=n_comp)
    pca_tr = pca_tmp.fit_transform(X_tr)
    pca_te = pca_tmp.transform(X_te)
    
    svm_tmp = SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42)
    svm_tmp.fit(pca_tr, train_labels)
    pred = svm_tmp.predict(pca_te)
    proba = svm_tmp.predict_proba(pca_te)[:, 1]
    
    feat_comparison[feat_name] = compute_all_metrics(test_labels, pred, proba)

feat_df = pd.DataFrame(feat_comparison).T.round(4)
print(feat_df.to_string())
feat_df.to_csv(os.path.join(RESULTS_DIR, 'feature_comparison.csv'))

# Plot
fig, ax = plt.subplots(figsize=(14, 3))
ax.axis('off')
table = ax.table(cellText=feat_df.values, rowLabels=feat_df.index,
                 colLabels=feat_df.columns, cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.5)
for j in range(len(feat_df.columns)):
    table[0, j].set_facecolor('#2ecc71')
    table[0, j].set_text_props(color='white', fontweight='bold')
plt.title('Feature Type Comparison (SVM RBF Classifier)', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'feature_comparison_table.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Pipeline Visualization & Feature Importance

In [ ]:
# End-to-end pipeline example
fig, axes = plt.subplots(4, 5, figsize=(25, 20))

# Select 2 Normal + 2 Pneumonia from test set
normal_test_idx = np.where(test_labels == 0)[0][:2]
pneumonia_test_idx = np.where(test_labels == 1)[0][:2]
demo_indices = np.concatenate([normal_test_idx, pneumonia_test_idx])

for row, idx in enumerate(demo_indices):
    img = load_image(test_files[idx])
    true_label = 'Normal' if test_labels[idx] == 0 else 'Pneumonia'
    pred_label = 'Normal' if all_predictions['SVM (RBF)'][idx] == 0 else 'Pneumonia'
    
    # Original
    axes[row, 0].imshow(img, cmap='gray')
    axes[row, 0].set_title(f'Original\n(True: {true_label})')
    axes[row, 0].axis('off')
    
    # Segmented
    roi, mask = segment_lung_roi(img)
    axes[row, 1].imshow(roi, cmap='gray')
    axes[row, 1].set_title('Segmented ROI')
    axes[row, 1].axis('off')
    
    # LBP
    lbp_img, _ = compute_lbp(roi)
    axes[row, 2].imshow(lbp_img, cmap='gray')
    axes[row, 2].set_title('LBP Features')
    axes[row, 2].axis('off')
    
    # HOG
    _, hog_img = compute_hog_features(roi)
    axes[row, 3].imshow(hog_img, cmap='gray')
    axes[row, 3].set_title('HOG Features')
    axes[row, 3].axis('off')
    
    # Prediction
    color = '#2ecc71' if pred_label == true_label else '#e74c3c'
    axes[row, 4].text(0.5, 0.5, f'Predicted:\n{pred_label}',
                      transform=axes[row, 4].transAxes, fontsize=16,
                      verticalalignment='center', horizontalalignment='center',
                      bbox=dict(boxstyle='round', facecolor=color, alpha=0.8),
                      color='white', fontweight='bold')
    axes[row, 4].axis('off')

col_titles = ['Input Image', 'Otsu Segmentation', 'LBP Texture', 'HOG Gradient', 'Classification']
for col, title in enumerate(col_titles):
    axes[0, col].set_title(f'{title}\n{axes[0, col].get_title()}', fontsize=11, fontweight='bold')

plt.suptitle('End-to-End Pipeline: From Image to Classification', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'pipeline_example.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance from Random Forest
fig, ax = plt.subplots(figsize=(12, 6))

importances = rf.feature_importances_
top_k = 20
top_indices = np.argsort(importances)[-top_k:][::-1]
top_importances = importances[top_indices]

ax.barh(range(top_k), top_importances[::-1], color='#3498db', edgecolor='black')
ax.set_yticks(range(top_k))
ax.set_yticklabels([f'PCA Component {i+1}' for i in top_indices[::-1]])
ax.set_xlabel('Feature Importance')
ax.set_title('Top 20 Most Important PCA Components (Random Forest)', fontsize=14, fontweight='bold')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Summary & Save Results

In [ ]:
# Find best classifier
best_classifier = max(
    {k: v for k, v in all_results.items() if k != 'K-Means (Scratch)'},
    key=lambda x: all_results[x]['Accuracy']
)

print('=' * 70)
print('PHASE 2 - COMPLETE RESULTS SUMMARY')
print('=' * 70)

print(f'\n--- Dataset ---')
print(f'Training images: {len(train_files)} (Normal: {np.sum(train_labels==0)}, Pneumonia: {np.sum(train_labels==1)})')
print(f'Test images:     {len(test_files)} (Normal: {np.sum(test_labels==0)}, Pneumonia: {np.sum(test_labels==1)})')

print(f'\n--- Feature Extraction ---')
print(f'Handcrafted features: {X_train_hc.shape[1]} dims (LBP: 256, GLCM: 24, HOG: {X_train_hc.shape[1]-280})')
print(f'Best CNN model: {best_cnn} ({cnn_features_train[best_cnn].shape[1]} dims)')
print(f'Fused features: {X_train_fused.shape[1]} dims')
print(f'After PCA (95%% variance): {n_components} dims')

print(f'\n--- Best Classifier ---')
print(f'{best_classifier}:')
for metric, value in all_results[best_classifier].items():
    if value is not None:
        print(f'  {metric:15s}: {value:.4f}')

print(f'\n--- K-Means Clustering ---')
print(f'Test Accuracy: {kmeans_acc_test:.4f}')

print(f'\n--- From-Scratch Implementations ---')
print(f'  1. Otsu Thresholding (Segmentation)')
print(f'  2. Local Binary Pattern (Feature Extraction)')
print(f'  3. GLCM (Feature Extraction)')
print(f'  4. PCA (Dimensionality Reduction)')
print(f'  5. KNN (Classification)')
print(f'  6. K-Means (Clustering)')

print(f'\n--- Saved Outputs ---')
print(f'Figures: {FIGURES_DIR}')
print(f'Results: {RESULTS_DIR}')

In [ ]:
# Save all results
results_package = {
    'classification_results': all_results,
    'cnn_comparison': cnn_comparison if 'cnn_comparison' in dir() else cnn_accuracies,
    'feature_comparison': feat_comparison,
    'best_classifier': best_classifier,
    'best_cnn': best_cnn,
    'pca_n_components': n_components,
    'pca_variance_explained': float(sum(pca.explained_variance_ratio)),
    'kmeans_accuracy': kmeans_acc_test,
    'best_k_knn': best_k,
    'train_size': len(train_files),
    'test_size': len(test_files),
}

with open(os.path.join(RESULTS_DIR, 'phase2_results.pkl'), 'wb') as f:
    pickle.dump(results_package, f)

print('All results saved to:', os.path.join(RESULTS_DIR, 'phase2_results.pkl'))
print('\nFigures saved:')
for f in sorted(os.listdir(FIGURES_DIR)):
    print(f'  {f}')

In [ ]:
# Save trained models for the web app
import joblib

models_dir = os.path.join(PROJECT_DIR, 'models')
os.makedirs(models_dir, exist_ok=True)

# Save scalers
joblib.dump(scaler_cnn, os.path.join(models_dir, 'scaler_cnn.pkl'))
joblib.dump(scaler_hc, os.path.join(models_dir, 'scaler_hc.pkl'))

# Save PCA (from scratch object)
with open(os.path.join(models_dir, 'pca_scratch.pkl'), 'wb') as f:
    pickle.dump(pca, f)

# Save classifiers
joblib.dump(svm_rbf, os.path.join(models_dir, 'svm_rbf.pkl'))
joblib.dump(svm_lin, os.path.join(models_dir, 'svm_linear.pkl'))
joblib.dump(rf, os.path.join(models_dir, 'random_forest.pkl'))
joblib.dump(xgb, os.path.join(models_dir, 'xgboost.pkl'))

# Save KNN training data (needed since KNN stores all training points)
np.save(os.path.join(models_dir, 'knn_train_X.npy'), X_train_pca)
np.save(os.path.join(models_dir, 'knn_train_y.npy'), train_labels)
joblib.dump(best_k, os.path.join(models_dir, 'knn_best_k.pkl'))

# Save config
config = {
    'best_cnn': best_cnn,
    'best_classifier': best_classifier,
    'best_k': best_k,
    'n_pca_components': n_components,
    'img_size': IMG_SIZE,
    'cnn_feature_dim': cnn_features_train[best_cnn].shape[1],
    'handcrafted_dim': X_train_hc.shape[1],
}
with open(os.path.join(models_dir, 'config.pkl'), 'wb') as f:
    pickle.dump(config, f)

print(f'All models saved to: {models_dir}')
for f_name in sorted(os.listdir(models_dir)):
    size_mb = os.path.getsize(os.path.join(models_dir, f_name)) / 1024 / 1024
    print(f'  {f_name}: {size_mb:.2f} MB')

In [ ]:
print('\n' + '=' * 70)
print('PHASE 2 COMPLETE')
print('=' * 70)
print('\nNext steps:')
print('1. Write the research paper using the figures and tables generated')
print('2. Create the PowerPoint presentation (10 minutes)')
print('3. All figures are saved in the figures/ directory')
print('4. All numerical results are saved in results/')